# Smart Downsize / Local Alternative Engine — Notebook

Fitur: ketika pengguna melihat produk brand internasional/mahal, sistem menyajikan
**alternatif UMKM lokal** yang spesifikasinya sepadan dengan harga jauh lebih murah
(target hemat 40-50%), lengkap dengan skor kualitas dari **Aspect-Based Sentiment
Analysis (ABSA)** atas ulasan pembeli.

Definisi problem & justifikasi algoritma lengkap ada di
`01_PROBLEM_DEFINITION.md` bagian 6. Ringkas: ini problem **content-based
matching/ranking** (bukan klasifikasi/regresi) — TF-IDF + cosine similarity untuk
kecocokan spesifikasi, leksikon sentimen Bahasa Indonesia (rule-based, bukan model
terlatih) untuk skor kualitas per aspek.

> Sesuai ketentuan lomba: hanya core inference dengan parameter statis. Tidak ada
> auto-tuning, bulk testing script, atau feedback-loop otomatis.

In [1]:
import sys
sys.path.insert(0, "src")
import pandas as pd
import json

from generate_dummy_catalog import generate_dummy_catalog
from local_alternative_engine import (
    LocalAlternativeEngine, aspect_based_sentiment, overall_sentiment_score,
    spec_overlap_pct, build_catalog_text,
)
pd.set_option("display.max_colwidth", 60)

## 1. Dataset (dummy)

Data spesifikasi produk & ulasan hasil scraping asli belum cukup banyak untuk
mendemonstrasikan spec-matching & ABSA, jadi dipakai **katalog dummy**: 5 "archetype"
produk (charger, TWS earphone, sepatu, tas ransel, smartwatch) — tiap archetype punya
beberapa listing "brand internasional" (harga tinggi, official store, spesifikasi
lengkap) dan beberapa listing "UMKM lokal" (harga 40-60% lebih murah, sebagian
spesifikasi sama) + ulasan sintetis Bahasa Indonesia per listing.

In [2]:
catalog_path = "data/dummy_catalog.json"
try:
    df = pd.read_json(catalog_path)
except (FileNotFoundError, ValueError):
    df = generate_dummy_catalog()
    df.to_json(catalog_path, orient="records", force_ascii=False, indent=2)

print(df.shape)
df[["name", "brand_tier", "seller", "price", "rating", "total_reviews"]].head(10)

(33, 14)


,name,brand_tier,seller,price,rating,total_reviews
0,Charger USB-C 20W Fast Charging - UrbanFit World Original,International,UrbanFit World Official Store,329000,4.6,318
1,Charger USB-C 20W Fast Charging - ProGear Intl Original,International,ProGear Intl Official Store,274000,4.7,222
2,Charger USB-C 20W Fast Charging - Versi Lokal Hemat,Lokal_UMKM,Griya Charger Nusantara,191000,4.8,181
3,Charger USB-C 20W Fast Charging - Versi Lokal Premium,Lokal_UMKM,UMKM Charger Bersaudara,179000,4.4,19
4,Charger USB-C 20W Fast Charging - Versi Lokal Best Seller,Lokal_UMKM,Griya Charger Bersaudara,193000,4.4,347
5,Charger USB-C 20W Fast Charging - Versi Lokal Premium,Lokal_UMKM,UMKM Charger Bersaudara,166000,4.8,139
6,Charger USB-C 20W Fast Charging - Versi Lokal Best Seller,Lokal_UMKM,Kriya Charger Jaya,186000,4.3,110
7,TWS Earphone Bluetooth 5.3 ANC - UrbanFit World Original,International,UrbanFit World Official Store,1263000,4.8,179
8,TWS Earphone Bluetooth 5.3 ANC - Versi Lokal Hemat,Lokal_UMKM,Toko TWS Sejahtera,750000,4.4,72
9,TWS Earphone Bluetooth 5.3 ANC - Versi Lokal Hemat,Lokal_UMKM,Sentra TWS Mandiri,666000,4.4,205


In [3]:
df.groupby(["archetype", "brand_tier"])["price"].agg(["count", "mean"]).round(0)

count       mean
archetype                       brand_tier                     
Charger USB-C 20W Fast Charging International      2   301500.0
                                Lokal_UMKM         5   183000.0
Jam Tangan Smartwatch Fitness   International      2  2412500.0
                                Lokal_UMKM         5  1232800.0
Sepatu Sneakers Running         International      1  1644000.0
                                Lokal_UMKM         5   881000.0
TWS Earphone Bluetooth 5.3 ANC  International      1  1263000.0
                                Lokal_UMKM         5   769200.0
Tas Ransel Laptop Anti Air      International      2   952000.0
                                Lokal_UMKM         5   471800.0

## 2. Preprocessing: Teks Katalog untuk Spec Matching

Gabungkan nama + kategori + teks spesifikasi jadi satu field teks per produk
(`build_catalog_text`), lalu di-vectorize dengan TF-IDF (unigram+bigram).

In [4]:
sample_text = build_catalog_text(df).iloc[0]
print("Contoh teks katalog (setelah digabung):")
print(sample_text)

Contoh teks katalog (setelah digabung):
Charger USB-C 20W Fast Charging - UrbanFit World Original Aksesoris Gadget USB-C, 20W, fast charging, power delivery, PD, kompatibel iPhone, kompatibel Samsung, kabel data termasuk, garansi resmi 1 tahun, original


## 3. "Training" — Fit TF-IDF Vectorizer

Untuk pipeline berbasis *retrieval* seperti ini, tidak ada target label untuk
supervised training. "Training" di sini berarti **fit vectorizer sekali ke seluruh
korpus katalog** (statis) — parameter vocabulary & IDF weight dihitung sekali dan
dipakai berulang untuk semua pencarian alternatif berikutnya, tanpa refit per
request.

In [5]:
engine = LocalAlternativeEngine().fit(df)
print(f"Ukuran vocabulary TF-IDF: {len(engine.vectorizer.vocabulary_)} term")
print(f"Ukuran matrix TF-IDF: {engine.tfidf_matrix.shape}")

Ukuran vocabulary TF-IDF: 355 term
Ukuran matrix TF-IDF: (33, 355)


## 4. Cek Leksikon ABSA (sanity check)

Sebelum dipakai untuk ranking, verifikasi dulu leksikon sentimen bekerja masuk akal
pada beberapa contoh ulasan buatan sendiri.

In [6]:
contoh_ulasan_positif = [
    "Kualitasnya bagus banget, awet dipakai tiap hari.",
    "Pengiriman cepat, packing rapi dan aman.",
    "Harga murah tapi kualitas oke, worth it banget.",
]
contoh_ulasan_negatif = [
    "Kualitas kurang bagus, baru sebulan udah rusak.",
    "Pengiriman lambat banget dan packing kurang aman.",
]

print("Ulasan positif ->", aspect_based_sentiment(contoh_ulasan_positif))
print()
print("Ulasan negatif ->", aspect_based_sentiment(contoh_ulasan_negatif))
print()
print("Skor keseluruhan (positif):", overall_sentiment_score(contoh_ulasan_positif))
print("Skor keseluruhan (negatif):", overall_sentiment_score(contoh_ulasan_negatif))

Ulasan positif -> {'Kualitas': {'score': 1.0, 'mentions': 2}, 'Keawetan': {'score': 1.0, 'mentions': 1}, 'Pengiriman': {'score': 1.0, 'mentions': 1}, 'Pelayanan': {'score': 0.0, 'mentions': 0}, 'Harga': {'score': 1.0, 'mentions': 1}}

Ulasan negatif -> {'Kualitas': {'score': -0.33, 'mentions': 1}, 'Keawetan': {'score': -0.33, 'mentions': 1}, 'Pengiriman': {'score': -0.33, 'mentions': 1}, 'Pelayanan': {'score': 0.0, 'mentions': 0}, 'Harga': {'score': 0.0, 'mentions': 0}}

Skor keseluruhan (positif): 1.0
Skor keseluruhan (negatif): -0.33


## 5. Evaluasi: Uji Pipeline pada Beberapa Produk Sumber

Karena ini problem *ranking* tanpa ground-truth label, evaluasi dilakukan dengan
**pengecekan manual terhadap kriteria bisnis** untuk tiap archetype produk:
1. Kategori alternatif = kategori sumber?
2. Hemat harga ada di kisaran target (≥25%, idealnya 40-50%)?
3. Rating & sentimen alternatif cukup baik (tidak merekomendasikan produk jelek)?
4. Kecocokan spesifikasi masuk akal (bukan produk yang sama sekali beda)?

In [7]:
def evaluate_source(source_row, top_k=3):
    alts = engine.find_alternatives(
        source_name=source_row["name"],
        source_category=source_row["category"],
        source_specifications_text=source_row["specifications_text"],
        source_price=source_row["price"],
        top_k=top_k,
    )
    print(f"=== SUMBER: {source_row['name']} — Rp{source_row['price']:,} ({source_row['category']}) ===")
    if not alts:
        print("  (tidak ada alternatif lolos threshold)")
        return []
    for a in alts:
        print(f"  -> {a['name']}  |  {a['seller']}")
        print(f"     Rp{a['price']:,}  (hemat {a['price_savings_pct']}%)  "
              f"| spec match {a['spec_match_pct']}%  | rating {a['rating']}  "
              f"| sentiment {a['sentiment_score']:+.2f} (aspek unggulan: {a['highlight_aspect']})")
    print()
    return alts

eval_summary = []
for archetype in df["archetype"].unique():
    source_row = df[(df.archetype == archetype) & (df.brand_tier == "International")].iloc[0]
    alts = evaluate_source(source_row)
    for a in alts:
        eval_summary.append({
            "archetype": archetype,
            "hemat_%": a["price_savings_pct"],
            "spec_match_%": a["spec_match_pct"],
            "rating": a["rating"],
            "sentiment": a["sentiment_score"],
        })

=== SUMBER: Charger USB-C 20W Fast Charging - UrbanFit World Original — Rp329,000 (Aksesoris Gadget) ===
  -> Charger USB-C 20W Fast Charging - Versi Lokal Hemat  |  Griya Charger Nusantara
     Rp191,000  (hemat 41.9%)  | spec match 58.8%  | rating 4.8  | sentiment +0.74 (aspek unggulan: Pengiriman)
  -> Charger USB-C 20W Fast Charging - Versi Lokal Premium  |  UMKM Charger Bersaudara
     Rp166,000  (hemat 49.5%)  | spec match 52.9%  | rating 4.8  | sentiment +0.60 (aspek unggulan: Pengiriman)
  -> Charger USB-C 20W Fast Charging - Versi Lokal Premium  |  UMKM Charger Bersaudara
     Rp179,000  (hemat 45.6%)  | spec match 52.9%  | rating 4.4  | sentiment +0.78 (aspek unggulan: Kualitas)

=== SUMBER: TWS Earphone Bluetooth 5.3 ANC - UrbanFit World Original — Rp1,263,000 (Aksesoris Gadget) ===
  -> TWS Earphone Bluetooth 5.3 ANC - Versi Lokal Hemat  |  UMKM TWS Sejahtera
     Rp776,000  (hemat 38.6%)  | spec match 61.1%  | rating 4.8  | sentiment +1.00 (aspek unggulan: Pelayanan)
  -> 

In [8]:
summary_df = pd.DataFrame(eval_summary)
print("Ringkasan seluruh rekomendasi yang dihasilkan:")
print(summary_df.describe().round(1))
print()
print(f"Rata-rata hemat harga : {summary_df['hemat_%'].mean():.1f}%  (target 40-50%)")
print(f"Rata-rata rating alternatif : {summary_df['rating'].mean():.2f} / 5.0")
print(f"% rekomendasi dgn sentimen positif (>0) : {(summary_df['sentiment'] > 0).mean()*100:.0f}%")

Ringkasan seluruh rekomendasi yang dihasilkan:
       hemat_%  spec_match_%  rating  sentiment
count     15.0          15.0    15.0       15.0
mean      46.5          55.1     4.5        0.7
std        5.4          11.3     0.3        0.2
min       38.6          36.8     3.9        0.1
25%       41.2          48.7     4.4        0.6
50%       47.0          55.0     4.5        0.7
75%       49.8          60.6     4.6        0.8
max       56.3          75.0     4.8        1.0

Rata-rata hemat harga : 46.5%  (target 40-50%)
Rata-rata rating alternatif : 4.48 / 5.0
% rekomendasi dgn sentimen positif (>0) : 100%


**Interpretasi hasil**: rata-rata hemat harga ada di kisaran target (40-50%),
mayoritas rekomendasi punya rating baik & sentimen ulasan positif — artinya pipeline
tidak asal merekomendasikan produk murah tapi jelek. Angka *spec match* di dataset
dummy ini masih di kisaran 40-65% (bukan 80-90% persis) karena pengukurannya
token-level yang ketat; ini area kalibrasi lanjutan begitu data spesifikasi asli
hasil scraping tersedia (lihat catatan di `01_PROBLEM_DEFINITION.md`).

## 6. Demo Inference — Kasus Sesuai Deskripsi Fitur

Simulasi persis skenario di deskripsi fitur: pengguna melihat **aksesoris gadget
brand internasional yang mahal**, sistem menyajikan alternatif UMKM lokal.

In [9]:
source = df[(df.archetype == "TWS Earphone Bluetooth 5.3 ANC") & (df.brand_tier == "International")].iloc[0]

result = engine.find_alternatives(
    source_name=source["name"],
    source_category=source["category"],
    source_specifications_text=source["specifications_text"],
    source_price=source["price"],
    top_k=3,
)

output = {
    "produk_dilihat_user": {
        "nama": source["name"],
        "harga": int(source["price"]),
        "kategori": source["category"],
    },
    "alternatif_lokal_umkm": result,
}
print(json.dumps(output, indent=2, ensure_ascii=False))

{
  "produk_dilihat_user": {
    "nama": "TWS Earphone Bluetooth 5.3 ANC - UrbanFit World Original",
    "harga": 1263000,
    "kategori": "Aksesoris Gadget"
  },
  "alternatif_lokal_umkm": [
    {
      "name": "TWS Earphone Bluetooth 5.3 ANC - Versi Lokal Hemat",
      "seller": "UMKM TWS Sejahtera",
      "price": 776000,
      "price_savings_pct": 38.6,
      "spec_match_pct": 61.1,
      "rating": 4.8,
      "sentiment_score": 1.0,
      "aspect_sentiment": {
        "Kualitas": {
          "score": 1.0,
          "mentions": 1
        },
        "Keawetan": {
          "score": 0.0,
          "mentions": 0
        },
        "Pengiriman": {
          "score": 0.0,
          "mentions": 0
        },
        "Pelayanan": {
          "score": 1.0,
          "mentions": 2
        },
        "Harga": {
          "score": 1.0,
          "mentions": 2
        }
      },
      "highlight_aspect": "Pelayanan"
    },
    {
      "name": "TWS Earphone Bluetooth 5.3 ANC - Versi Lokal Hemat",

## 7. Ringkasan

- **Problem AI**: content-based matching/ranking (bukan klasifikasi/regresi) — cari &
  rangking produk UMKM lokal yang spesifikasinya mirip & harganya jauh lebih murah
  dari produk brand internasional yang sedang dilihat pengguna.
- **Algoritma**: TF-IDF + cosine similarity (spec matching) + ABSA leksikon Bahasa
  Indonesia (kualitas produk dari ulasan) + skor gabungan berbobot statis.
- **Hasil di dataset dummy**: hemat harga rata-rata sesuai target 40-50%, mayoritas
  rekomendasi punya rating & sentimen baik.
- **Langkah lanjut**: kalibrasi ulang threshold pakai data spesifikasi & ulasan hasil
  scraping asli; pertimbangkan sentence-embedding untuk spec matching yang lebih
  toleran terhadap variasi kata kalau presisi TF-IDF dirasa kurang.